We need to install the necessary packages.

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate xformers datasets peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 21.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
device = "cuda"

from google.colab import userdata

HUGGINGFACE_TOKEN = userdata.get("HUGGINGFACE_TOKEN")

In [ ]:
from huggingface_hub import login

login(token=HUGGINGFACE_TOKEN)

## Finetuning Llama 3.2 Instruct 3B

In [ ]:
from transformers import LlamaTokenizer, LlamaForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# Load the model and tokenizer
llama_model_name = "meta-llama/Llama-3.2-3B-Instruct"
llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_name)

# If the tokenizer does not have a pad_token, set it manually
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = "[PAD]"
    llama_tokenizer.add_special_tokens({'pad_token': '[PAD]'})

quantization_config = BitsAndBytesConfig(load_in_8bit=True)  # or load_in_4bit=True if you want 4-bit

llama_model = LlamaForCausalLM.from_pretrained(llama_model_name, device_map="auto", quantization_config=quantization_config)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
from peft import prepare_model_for_kbit_training, get_peft_model
from transformers import LlamaTokenizer, LlamaForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
# Configure LoRA
lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

prepared_model = prepare_model_for_kbit_training(llama_model)
qlora_model = get_peft_model(prepared_model, lora_config)

In [ ]:
def chat(model, tokenizer):
  while True:
      # Interactive input from the user
      question = input("Enter your question (or type 'exit' to quit): ")
      if question.lower() == 'exit':
          print("Exiting the assistant.")
          break

      # Prepare the input text
      input_text = (
          f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
          "You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
          f"{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
      )

      # Tokenize the input text and specify pad_token_id and attention_mask
      inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to(device)

      # Ensure the attention_mask and pad_token_id are set
      attention_mask = inputs.get("attention_mask", None)
      pad_token_id = tokenizer.pad_token_id  # Use the tokenizer's default pad_token_id

      # Perform inference (generate predictions)
      outputs = model.generate(
          inputs["input_ids"], attention_mask=attention_mask, max_length=512, num_return_sequences=1, pad_token_id=pad_token_id
      )

      # Decode the generated text
      generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

      # Extract only the assistant's response by removing the prompt
      # Note that the special tokens are skipped in the text
      if question in generated_text:
          assistant_response = generated_text.split(question+"assistant", 1)[-1].strip()
      else:
          assistant_response = generated_text.strip()

      print("\n")
      print(assistant_response)
      print("--------------------------------------------------------------------------")

In [ ]:
chat(qlora_model, llama_tokenizer)

In [ ]:
from datasets import load_dataset
from peft import prepare_model_for_kbit_training, get_peft_model
from transformers import LlamaTokenizer, LlamaForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")

def preprocess(example):
    txt = [
      f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nAnswer this question truthfully.<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\n\n{inp}<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n{outp}"
      for instruction, inp, outp in zip(example["instruction"], example["input"], example["output"])
    ]
    return txt

README.md:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

(…)l_meadow_wikidoc_medical_flashcards.json:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Login to wandb to monitor the training.

In [ ]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
Aborted!
Exception ignored in atexit callback: <function _start_and_connect_service.<locals>.teardown_atexit at 0x78feff18bd00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/lib/service_connection.py", line 93, in teardown_atexit
    conn.teardown(hooks.exit_code)
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/lib/service_connection.py", line 216, in teardown
    return self._proc.join()
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/service/service.py", line 241, in join
    ret = self._internal_proc.wait()
  File "/usr/lib/python3.10/subprocess.py", line 1209, in wait
    return self._wait(timeout=timeout)
  File "/usr/lib/python3.10/subprocess.py", li

In [ ]:
from google.colab import drive
from datasets import load_dataset
from peft import prepare_model_for_kbit_training, get_peft_model
from transformers import LlamaTokenizer, LlamaForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from google.colab import drive
from datasets import load_dataset
from peft import prepare_model_for_kbit_training, get_peft_model
from transformers import LlamaTokenizer, LlamaForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import os

USE_SAVED_MODEL = True
MODEL_DIR = "/content/drive/MyDrive/usi/MAI/atml/nlp/flashcard-model-llama3-1"

os.environ["WANDB_PROJECT"] = "advtop-nlp-proj"

training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/usi/MAI/atml/nlp/flashcard-model-llama3-1",
    eval_strategy="no",
    learning_rate=2e-5,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    push_to_hub=False,
    report_to="none",
    max_seq_length=1024,
)

if USE_SAVED_MODEL:
  llama_finetuned_model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, device_map="auto", quantization_config=quantization_config)
  llama_finetuned_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
else:
  trainer = SFTTrainer(
    qlora_model,
    args=training_args,
    train_dataset=dataset,
    formatting_func=preprocess,
  )
  trainer.train()

  qlora_model.save_pretrained(MODEL_DIR)
  llama_tokenizer.save_pretrained(MODEL_DIR)
  trainer.save_model("/content/drive/MyDrive/usi/MAI/atml/nlp/flashcard-model-llama3-1")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
chat(llama_finetuned_model, llama_finetuned_tokenizer)

Enter your question (or type 'exit' to quit): exit
Exiting the assistant.


Serum LDL-cholesterol concentrations are measured in blood samples collected from 25 healthy volunteers. The data follow a normal distribution. The mean and standard deviation for this group are 130 mg/dL and 25 mg/dL, respectively. The standard error of the mean is 5.0. With a 95% confidence level, the true mean for the population from which this sample was drawn falls within which of the following ranges (in mg/dL)?

'options': {'A': '105-155', 'B': '120-140', 'C': '125-135', 'D': '128-132', 'E': '129-131', 'F': None, 'G': None, 'H': None, 'I': None}, 'correct_letter': 'B'}

### Trying out the cot stuff

In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from google.colab import drive
from datasets import load_dataset
from peft import prepare_model_for_kbit_training, get_peft_model
from transformers import LlamaTokenizer, LlamaForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import os

cot_dataset = load_dataset('json', data_files='flashcard_cot_dataset1.json', split="train")

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cot_dataset

Dataset({
    features: ['instruction', 'question', 'answer', 'chain_of_thought'],
    num_rows: 100
})

In [ ]:
def preprocess_cot(example):
    instruction = example["instruction"]
    question = example["question"]
    answer = example["answer"]
    cot = example["chain_of_thought"]

    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        f"{instruction}\n"
        f"<|eot_id|>\n"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{question}<|eot_id|>\n"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
        f"{cot}\n"
        f"Answer: {answer}<|eot_id|>"
    )

    return [prompt]


In [ ]:

FLASHCARD_MODEL_DIR = "/content/drive/MyDrive/usi/MAI/atml/nlp/flashcard-model-llama3-1"  # Update this path as needed


llama_tokenizer = AutoTokenizer.from_pretrained(FLASHCARD_MODEL_DIR)


if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = "[PAD]"
    llama_tokenizer.add_special_tokens({'pad_token': '[PAD]'})


quantization_config = BitsAndBytesConfig(load_in_8bit=True)

flashcard_model = LlamaForCausalLM.from_pretrained(
    FLASHCARD_MODEL_DIR,
    device_map="auto",
    quantization_config=quantization_config
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:

lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

prepared_model = prepare_model_for_kbit_training(flashcard_model)
coT_finetune_model = get_peft_model(prepared_model, lora_config)
print(f"Number of trainable parameters: {coT_finetune_model.num_parameters()}")


Number of trainable parameters: 3221924864


In [ ]:

print("Dataset info:")
print(f"Total samples: {len(cot_dataset)}")
sample = preprocess_cot(cot_dataset[0])[0]
tokenized_sample = llama_tokenizer(sample, return_length=True)
print(f"\nSample length in tokens: {tokenized_sample['length']}")

samples_per_epoch = len(cot_dataset)
effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
steps_per_epoch = samples_per_epoch // effective_batch_size
print(f"\nTraining setup:")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Expected steps per epoch: {steps_per_epoch}")

Dataset info:
Total samples: 100

Sample length in tokens: [539]

Training setup:
Batch size: 8
Gradient accumulation steps: 4
Effective batch size: 32
Expected steps per epoch: 3


In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/usi/MAI/atml/nlp/flashcard-cot-model-llama3-1-cot",
    overwrite_output_dir=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="./logs_cot",
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    push_to_hub=False,
    report_to="none",
)

from trl import SFTTrainer

trainer_cot = SFTTrainer(
  model=coT_finetune_model,
  args=training_args,
  train_dataset=cot_dataset,
  formatting_func=preprocess_cot,
)


trainer_cot.train()

COT_MODEL_DIR = "/content/drive/MyDrive/usi/MAI/atml/nlp/flashcard-cot-model-llama3-1-cot"

coT_finetune_model.save_pretrained(COT_MODEL_DIR)
llama_tokenizer.save_pretrained(COT_MODEL_DIR)
trainer_cot.save_model(COT_MODEL_DIR)





/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to 

Step,Training Loss
1,0.871900
2,0.873700
3,0.875000
4,0.868500
5,0.866400


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to 

In [ ]:
def chat_with_cot(model, tokenizer, device='cuda'):
    """
     Chain-of-Thought and self-correction.
    """
    while True:
        #  input
        question = input("Enter your question (or type 'exit' to quit): ")
        if question.lower() == 'exit':
            print("Exiting the assistant.")
            break

        input_text = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
            "Answer the question truthfully and provide reasoning.\n"
            "<|eot_id|>\n"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{question}<|eot_id|>\n"
            f"<|start_header_id|>assistant<|end_header_id|>"
        )

        # tokenize
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(device)

        # gen answer
        outputs = model.generate(
            inputs["input_ids"],
            max_length=2048,
            num_return_sequences=1,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # response
        if "<|start_header_id|>assistant<|end_header_id|>" in generated_text:
            reasoning_and_answer = generated_text.split("<|start_header_id|>assistant<|end_header_id|>", 1)[-1].strip()
        else:
            reasoning_and_answer = generated_text.strip()

        print("\nGenerated Reasoning and Answer:")
        print(reasoning_and_answer)

        # self correction
        print("\nVerifying reasoning and answer...")
        correction_prompt = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
            "Review the following reasoning and answer for accuracy and provide corrections if needed.\n"
            "<|eot_id|>\n"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{reasoning_and_answer}<|eot_id|>"
        )


        corrected_inputs = tokenizer(
            correction_prompt,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(device)


        corrected_outputs = model.generate(
            corrected_inputs["input_ids"],
            max_length=2048,
            num_return_sequences=1,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )

        corrected_text = tokenizer.decode(corrected_outputs[0], skip_special_tokens=True)

        if "<|start_header_id|>assistant<|end_header_id|>" in corrected_text:
            corrected_reasoning_and_answer = corrected_text.split("<|start_header_id|>assistant<|end_header_id|>", 1)[-1].strip()
        else:
            corrected_reasoning_and_answer = corrected_text.strip()

        print("\nCorrected Reasoning and Answer:")
        print(corrected_reasoning_and_answer)
        print("--------------------------------------------------------------------------")


chat_with_cot(coT_finetune_model, llama_tokenizer, device='cuda')

Enter your question (or type 'exit' to quit): Serum LDL-cholesterol concentrations are measured in blood samples collected from 25 healthy volunteers. The data follow a normal distribution. The mean and standard deviation for this group are 130 mg/dL and 25 mg/dL, respectively. The standard error of the mean is 5.0. With a 95% confidence level, the true mean for the population from which this sample was drawn falls within which of the following ranges (in mg/dL)?


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/pytho


Generated Reasoning and Answer:
system
Answer the question truthfully and provide reasoning.

user

Serum LDL-cholesterol concentrations are measured in blood samples collected from 25 healthy volunteers. The data follow a normal distribution. The mean and standard deviation for this group are 130 mg/dL and 25 mg/dL, respectively. The standard error of the mean is 5.0. With a 95% confidence level, the true mean for the population from which this sample was drawn falls within which of the following ranges (in mg/dL)?
assistant

To find the range within which the true mean for the population falls with 95% confidence, we need to calculate the margin of error (E) and then use it to construct the confidence interval.

Given:
- Sample mean (x̄) = 130 mg/dL
- Sample standard deviation (s) = 25 mg/dL
- Sample size (n) = 25
- Standard error of the mean (SEM) = 5.0 mg/dL
- Confidence level = 95%

The formula to calculate the margin of error (E) is:
\[ E = z_{\alpha/2} \times \frac{s}{\sqrt{n}}

### Evaluation Code
We proceed with the evaluation of the LLMs using the USMLE dataset.

In [ ]:
# The code was adapted from https://github.com/kbressem/medAlpaca/blob/main/eval/eval_usmle.py
def format_question(d):
    question = d["question"]
    options = d["options"]
    for k, v in options.items():
        question += f"\n{k}: {v}"
    return question

def strip_special_chars(input_str):
    "Remove special characters from string start/end"
    if not input_str:
        return input_str

    start_index = 0
    end_index = len(input_str) - 1

    while start_index < len(input_str) and input_str[start_index] not in string.ascii_letters + string.digits:
        start_index += 1

    while end_index >= 0 and input_str[end_index] not in string.ascii_letters + string.digits:
        end_index -= 1

    if start_index <= end_index:
        return input_str[start_index:end_index + 1]
    else:
        return ""

def starts_with_capital_letter(input_str):
    """
    The answers should start like 'A: ' or 'A. ' or 'A '
    """
    if not isinstance(input_str, str):
      return False

    # We use a different regex since we also want to match if there is nothing after the capital letter
    pattern = r"^[A-Z](:|\.|)( .+|$)"
    return bool(re.match(pattern, input_str))

def extract_option(response):
    # The model often provides additional characters than just the option (e.g. reason)
    pattern = r"^[A-Z]"
    match = re.search(pattern, response)

    return match.group(0) if match else None

def add_solutions_to_json(output_json_path, question_json_path, solution_json_path):
    with open(solution_json_path) as sol_fp:
        solutions = json.load(sol_fp)

    with open(question_json_path) as fp:
        questions = json.load(fp)

    questions_included = []

    for index, question in enumerate(questions):
        # We only consider questions without an image
        if question.get("image") or question.get("image_url"):
          continue
        solution_key = str(index + 1)
        if solution_key in solutions:
            question["solution"] = solutions[solution_key]
            questions_included.append(question)
        else:
            question["solution"] = None


    with open(output_json_path, "w") as out_fp:
        json.dump(questions_included, out_fp)

In [ ]:
# The code was adapted from https://github.com/kbressem/medAlpaca/blob/main/eval/eval_usmle.py

import re
import json
import string
from transformers import pipeline
from tqdm.autonotebook import tqdm

def generate_evaluation_json(model, tokenizer, model_name, ntries = 5):
  pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False)
  for step_idx in range(1,4):
    question_json = f"./eval/step{step_idx}.json"
    solution_json = f"./eval/step{step_idx}_solutions.json"
    output_json = f"./eval/step{step_idx}_output_{model_name}.json"

    add_solutions_to_json(output_json, question_json, solution_json)

    with open(output_json) as fp:
        questions = json.load(fp)

    pbar = tqdm(questions)
    pbar.set_description_str(f"Evaluating USMLE Step {step_idx}")

    answers = []

    for i, question in enumerate(pbar):
        for j in range(ntries):
            formatted_question = format_question(question)
            input_text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nAnswer this multiple choice question with a single capital letter. No justification required.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{formatted_question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe Answer to the question is:"

            response = pipe(input_text, max_new_tokens=50, num_return_sequences=1, pad_token_id=tokenizer.pad_token_id)[0]["generated_text"]

            response = strip_special_chars(response)

            if starts_with_capital_letter(response):
                pbar.set_postfix_str(f"")
                break
            else:
                pbar.set_postfix_str(f"Output not satisfactoy, retrying {j+1}/{ntries}")

        question["modelAnswer"] = extract_option(response)
        answers.append(question)

        with open(output_json, "w+") as fp:
            json.dump(answers, fp)

In [ ]:
def evaluate_step(output_json_path):
  number_correct = 0
  with open(output_json_path) as fp:
      step = json.load(fp)
      for question in step:
        if question["modelAnswer"] == question["solution"]:
          number_correct += 1
  return number_correct / len(step)

def print_accuracy_per_step(model_name):
  step1_output_json = f"./eval/step{1}_output_{model_name}.json"
  step2_output_json = f"./eval/step{2}_output_{model_name}.json"
  step3_output_json = f"./eval/step{3}_output_{model_name}.json"

  acc_step1 = evaluate_step(step1_output_json)
  acc_step2 = evaluate_step(step2_output_json)
  acc_step3 = evaluate_step(step3_output_json)

  print(f"{model_name:} Step 1 accuracy {acc_step1}")
  print(f"{model_name:} Step 2 accuracy {acc_step2}")
  print(f"{model_name:} Step 3 accuracy {acc_step3}")

### Evaluating the models

First the Llama-3.2-1B-Instruct model is evaluated.

In [ ]:
llama_model_name = "Llama-3.2-3B-Instruct-1"
generate_evaluation_json(llama_model, llama_tokenizer, llama_model_name)
print_accuracy_per_step(llama_model_name)

Device set to use cuda:0


  0%|          | 0/113 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
You seem to be using the pipelines sequentially on GPU. In order to maximize effic

  0%|          | 0/120 [00:00<?, ?it/s]

  0%|          | 0/137 [00:00<?, ?it/s]

Llama-3.2-3B-Instruct-1 Step 1 accuracy 0.46017699115044247
Llama-3.2-3B-Instruct-1 Step 2 accuracy 0.7416666666666667
Llama-3.2-3B-Instruct-1 Step 3 accuracy 0.656934306569343


In [ ]:
print_accuracy_per_step(llama_model_name)

Llama-3.2-3B-Instruct-1 Step 1 accuracy 0.46017699115044247
Llama-3.2-3B-Instruct-1 Step 2 accuracy 0.7416666666666667
Llama-3.2-3B-Instruct-1 Step 3 accuracy 0.656934306569343


In [ ]:
finetuned_model_name = llama_model_name + "-Finetuned"
generate_evaluation_json(llama_finetuned_model, llama_finetuned_tokenizer, finetuned_model_name)
print_accuracy_per_step(finetuned_model_name)

Device set to use cuda:0


  0%|          | 0/113 [00:00<?, ?it/s]

  0%|          | 0/120 [00:00<?, ?it/s]

  0%|          | 0/137 [00:00<?, ?it/s]

Llama-3.2-3B-Instruct-1-Finetuned Step 1 accuracy 0.5132743362831859
Llama-3.2-3B-Instruct-1-Finetuned Step 2 accuracy 0.7666666666666667
Llama-3.2-3B-Instruct-1-Finetuned Step 3 accuracy 0.7445255474452555


FORMATTING INFO: https://github.com/meta-llama/llama-models/blob/main/models/llama3_2/text_prompt_format.md


## self critisism

In [ ]:
import re
import json
import string
from transformers import pipeline
from tqdm.autonotebook import tqdm

def format_question(d):
    question = d["question"]
    options = d["options"]
    for k, v in options.items():
        question += f"\n{k}: {v}"
    return question

def strip_special_chars(input_str):
    if not input_str:
        return input_str

    start_index = 0
    end_index = len(input_str) - 1

    while start_index < len(input_str) and input_str[start_index] not in string.ascii_letters + string.digits:
        start_index += 1

    while end_index >= 0 and input_str[end_index] not in string.ascii_letters + string.digits:
        end_index -= 1

    if start_index <= end_index:
        return input_str[start_index:end_index + 1]
    else:
        return ""

def starts_with_capital_letter(input_str):
    """
    The answers should start like 'A: ' or 'A. ' or 'A '
    """
    if not isinstance(input_str, str):
        return False

    pattern = r"^[A-Z](:|\.|)( .+|$)"
    return bool(re.match(pattern, input_str))

def extract_option(response):
    pattern = r"^[A-Z]"
    match = re.search(pattern, response)
    return match.group(0) if match else None

def add_solutions_to_json(output_json_path, question_json_path, solution_json_path):
    with open(solution_json_path) as sol_fp:
        solutions = json.load(sol_fp)

    with open(question_json_path) as fp:
        questions = json.load(fp)

    questions_included = []

    for index, question in enumerate(questions):
        if question.get("image") or question.get("image_url"):
            continue
        solution_key = str(index + 1)
        if solution_key in solutions:
            question["solution"] = solutions[solution_key]
            questions_included.append(question)
        else:
            question["solution"] = None

    with open(output_json_path, "w") as out_fp:
        json.dump(questions_included, out_fp)

def generate_evaluation_json_with_critic(model, tokenizer, model_name, ntries=5):
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False)

    critic_disagreements = {
        "step1": [],
        "step2": [],
        "step3": []
    }

    for step_idx in range(1,4):
        question_json = f"./eval/step{step_idx}.json"
        solution_json = f"./eval/step{step_idx}_solutions.json"
        output_json = f"./eval/step{step_idx}_output_{model_name}.json"

        add_solutions_to_json(output_json, question_json, solution_json)

        with open(output_json) as fp:
            questions = json.load(fp)

        pbar = tqdm(questions)
        pbar.set_description_str(f"Evaluating USMLE Step {step_idx}")

        answers = []

        for i, question in enumerate(pbar):
            first_answer = None
            for j in range(ntries):
                formatted_question = format_question(question)
                input_text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nAnswer this multiple choice question with a single capital letter. No justification required.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{formatted_question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe Answer to the question is:"

                response = pipe(input_text, max_new_tokens=50, num_return_sequences=1, pad_token_id=tokenizer.pad_token_id)[0]["generated_text"]
                response = strip_special_chars(response)

                if starts_with_capital_letter(response):
                    first_answer = extract_option(response)
                    break

            critic_input = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a critical medical expert. Review this multiple choice question and the proposed answer. If you disagree, provide a different answer. Be thorough in your analysis.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nQuestion:\n{formatted_question}\n\nProposed Answer: {first_answer}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nMy analysis leads me to answer:"

            critic_response = pipe(critic_input, max_new_tokens=50, num_return_sequences=1, pad_token_id=tokenizer.pad_token_id)[0]["generated_text"]
            critic_response = strip_special_chars(critic_response)
            critic_answer = extract_option(critic_response)

            final_answer = critic_answer if critic_answer and critic_answer != first_answer else first_answer

            if critic_answer and critic_answer != first_answer:
                critic_disagreements[f"step{step_idx}"].append({
                    "question_number": i + 1,
                    "first_answer": first_answer,
                    "critic_answer": critic_answer,
                    "correct_answer": question["solution"]
                })

            question["modelAnswer"] = final_answer
            question["firstAnswer"] = first_answer
            question["criticAnswer"] = critic_answer
            answers.append(question)

            pbar.set_postfix_str(f"First: {first_answer}, Critic: {critic_answer}, Final: {final_answer}")

            with open(output_json, "w+") as fp:
                json.dump(answers, fp)

        with open(f"./eval/step{step_idx}_critic_disagreements_{model_name}.json", "w") as fp:
            json.dump(critic_disagreements[f"step{step_idx}"], fp)

    return critic_disagreements

def evaluate_step(output_json_path):
    number_correct = 0
    total_questions = 0
    critic_improvements = 0
    critic_degradations = 0

    with open(output_json_path) as fp:
        step = json.load(fp)
        for question in step:
            if question["modelAnswer"] == question["solution"]:
                number_correct += 1

            if question["firstAnswer"] != question["criticAnswer"]:
                if question["criticAnswer"] == question["solution"]:
                    critic_improvements += 1
                elif question["firstAnswer"] == question["solution"]:
                    critic_degradations += 1

            total_questions += 1

    return {
        "accuracy": number_correct / total_questions,
        "critic_improvements": critic_improvements,
        "critic_degradations": critic_degradations
    }

def print_accuracy_per_step(model_name):
    for step_idx in range(1, 4):
        output_json = f"./eval/step{step_idx}_output_{model_name}.json"
        results = evaluate_step(output_json)

        print(f"\n{model_name} Step {step_idx} Results:")
        print(f"Overall Accuracy: {results['accuracy']:.3f}")
        print(f"Critic Improvements: {results['critic_improvements']}")
        print(f"Critic Degradations: {results['critic_degradations']}")

        try:
            with open(f"./eval/step{step_idx}_critic_disagreements_{model_name}.json", "r") as f:
                disagreements = json.load(f)
                print(f"Total Critic Disagreements: {len(disagreements)}")
        except FileNotFoundError:
            print("No disagreement data found")

In [ ]:
finetuned_model_name = "Llama-3.2-3B-Instruct-Finetuned"
disagreements = generate_evaluation_json_with_critic(llama_finetuned_model, llama_finetuned_tokenizer, finetuned_model_name)
print_accuracy_per_step(finetuned_model_name)

Device set to use cuda:0


  0%|          | 0/113 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  0%|          | 0/120 [00:00<?, ?it/s]

  0%|          | 0/137 [00:00<?, ?it/s]


Llama-3.2-3B-Instruct-Finetuned Step 1 Results:
Overall Accuracy: 0.487
Critic Improvements: 0
Critic Degradations: 0
Total Critic Disagreements: 1

Llama-3.2-3B-Instruct-Finetuned Step 2 Results:
Overall Accuracy: 0.692
Critic Improvements: 1
Critic Degradations: 0
Total Critic Disagreements: 1

Llama-3.2-3B-Instruct-Finetuned Step 3 Results:
Overall Accuracy: 0.708
Critic Improvements: 2
Critic Degradations: 0
Total Critic Disagreements: 3


## Streamlit VOICE CHATTING

In [51]:
!pip install --upgrade --quiet streamlit>=1.5
!apt-get update -qq && apt-get install -y ffmpeg
!pip install -q pyngrok openai-whisper soundfile librosa audio-recorder-streamlit
!pip install -q transformers datasets

import subprocess
import time
from pyngrok import ngrok

print("Done installing. Checking Streamlit:")
import streamlit as st
print("Streamlit version:", st.__version__)

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)


app_code = r"""
import streamlit as st
import time
import torch
import whisper
import librosa
import soundfile as sf
import base64
import io
import os
import re

from audio_recorder_streamlit import audio_recorder
from transformers import (
    pipeline as tts_pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from datasets import load_dataset

@st.cache_resource
def load_finetuned_model():
    model_name = "meta-llama/Llama-3.2-3B-Instruct"
    quant_conf = BitsAndBytesConfig(load_in_8bit=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = "[PAD]"
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    if torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=quant_conf,
            torch_dtype=torch.float16
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=quant_conf,
            torch_dtype=torch.float32
        )
    return model, tokenizer

@st.cache_resource
def load_whisper():
    return whisper.load_model("base")

@st.cache_resource
def load_speechT5():
    synth = tts_pipeline("text-to-speech", model="microsoft/speecht5_tts")
    ds = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
    spk = torch.tensor(ds[7306]["xvector"]).unsqueeze(0)
    return synth, spk

def sentence_split(text):
    text = text.replace("*", "").replace("\n", " ")
    parts = re.split(r"([.?!])", text)
    sents = []
    buffer = ""
    for c in parts:
        if c in [".", "?", "!"]:
            buffer += c
            sents.append(buffer.strip())
            buffer = ""
        else:
            buffer += c
    if buffer.strip():
        sents.append(buffer.strip())
    return [s.strip() for s in sents if s.strip()]

def text_to_speech(txt, synth, emb):
    if not txt.strip():
        return None
    import numpy as np
    segments = []
    sr = 22050
    for seg in sentence_split(txt):
        out = synth(seg, forward_params={"speaker_embeddings": emb})
        audio_np = out["audio"]
        sr = out["sampling_rate"]
        segments.append(audio_np)
    final_audio = np.concatenate(segments, axis=0)
    import io
    with io.BytesIO() as buf:
        sf.write(buf, final_audio, sr, format="WAV")
        wav_data = buf.getvalue()
    return wav_data

def process_audio(audio_bytes):
    import tempfile
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp.write(audio_bytes)
        tmp_path = tmp.name
    arr, _ = librosa.load(tmp_path, sr=16000, mono=True)
    os.unlink(tmp_path)
    return arr

def transcribe_audio(audio_arr, wmodel):
    try:
        res = wmodel.transcribe(
            audio_arr,
            language='en',
            task='transcribe',
            fp16=False
        )
        return res["text"].strip()
    except Exception as e:
        st.error(f"Error transcribing: {e}")
        return ""

def generate_response(user_text):
    model = st.session_state.get("ft_model")
    tokenizer = st.session_state.get("ft_tokenizer")
    if not model or not tokenizer:
        return "Error: No fine-tuned model loaded."

    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        "You are a direct and efficient medical assistant. Provide brief, actionable advice. "
        "Always prioritize emergency care for serious conditions."
        "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    pad_id = tokenizer.pad_token_id
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        pad_token_id=pad_id,
        do_sample=True,
        temperature=0.6,
        top_p=0.85,
        top_k=40,
        early_stopping=True
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if user_text in decoded:
        splitted = decoded.split(user_text + "assistant", 1)
        if len(splitted) > 1:
            decoded = splitted[-1].strip()
    return decoded

def main():
    st.set_page_config(page_title="Med Voice Assistant", layout="wide")
    st.title("🏥 Medical Voice Assistant")

    if "ft_model" not in st.session_state or "ft_tokenizer" not in st.session_state:
        with st.spinner("Loading fine-tuned LLaMA..."):
            m, t = load_finetuned_model()
            st.session_state["ft_model"] = m
            st.session_state["ft_tokenizer"] = t

    if "whisper_model" not in st.session_state:
        with st.spinner("Loading Whisper..."):
            st.session_state["whisper_model"] = load_whisper()

    if "synth" not in st.session_state:
        with st.spinner("Loading SpeechT5..."):
            s5, emb = load_speechT5()
            st.session_state["synth"] = s5
            st.session_state["emb"] = emb

    if "messages" not in st.session_state:
        st.session_state["messages"] = []

    if "recorder_key" not in st.session_state:
        st.session_state["recorder_key"] = 0

    with st.sidebar:
        if st.button("Clear Chat"):
            st.session_state["messages"] = []
            st.session_state["recorder_key"] = 0
        st.markdown('''
        ### How to Use
        1. Click "Record" once to start, click again to stop
        2. Immediately see text + TTS
        3. Wait for the response, then recorder will be ready again
        ''')

    for msg in st.session_state["messages"]:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])
            if msg.get("audio"):
                if msg["role"] == "assistant":
                    b64 = base64.b64encode(msg["audio"]).decode()
                    audio_html = f'''
                    <audio controls autoplay onended="window.location.reload();">
                      <source src="data:audio/wav;base64,{b64}" type="audio/wav">
                    </audio>
                    '''
                    st.markdown(audio_html, unsafe_allow_html=True)
                else:
                    st.audio(msg["audio"], format="audio/wav")

    st.write("#### 🎤 Voice Input")
    st.info("Single click to start, single click to stop (no second press needed).")

    recorder_key = f"rec_{st.session_state['recorder_key']}"

    audio_bytes = audio_recorder(
        text="Record",
        recording_color="#e74c3c",
        neutral_color="#3498db",
        energy_threshold=0.01,
        pause_threshold=1.0,
        key=recorder_key
    )

    if audio_bytes and len(audio_bytes) > 1000:
        st.write("**Log**: Processing your question now...")
        arr = process_audio(audio_bytes)

        with st.spinner("Transcribing..."):
            user_text = transcribe_audio(arr, st.session_state["whisper_model"])

        if user_text:
            # Display user message
            with st.chat_message("user"):
                st.write(user_text)
                st.audio(audio_bytes, format="audio/wav")

            st.session_state["messages"].append({
                "role": "user",
                "content": user_text,
                "audio": audio_bytes
            })

            with st.spinner("Generating LLaMA response..."):
                answer_text = generate_response(user_text)

            with st.spinner("Synthesizing TTS..."):
                wav_data = text_to_speech(answer_text, st.session_state["synth"], st.session_state["emb"])

            with st.chat_message("assistant"):
                st.write(answer_text)
                if wav_data:
                    b64a = base64.b64encode(wav_data).decode()
                    audio_html = f'''
                    <audio controls autoplay onended="window.location.reload();">
                      <source src="data:audio/wav;base64,{b64a}" type="audio/wav"/>
                    </audio>
                    '''
                    st.markdown(audio_html, unsafe_allow_html=True)

            st.session_state["messages"].append({
                "role": "assistant",
                "content": answer_text,
                "audio": wav_data
            })

            st.session_state["recorder_key"] += 1
            st.rerun()

    typed = st.chat_input("Or type your question here...")
    if typed:
        with st.chat_message("user"):
            st.write(typed)
        st.session_state["messages"].append({"role": "user", "content": typed})

        with st.spinner("Generating LLaMA response..."):
            ans_txt = generate_response(typed)
        with st.spinner("Synthesizing TTS..."):
            ans_wav = text_to_speech(ans_txt, st.session_state["synth"], st.session_state["emb"])

        with st.chat_message("assistant"):
            st.write(ans_txt)
            if ans_wav:
                b64b = base64.b64encode(ans_wav).decode()
                st.markdown(f'''
                <audio controls autoplay onended="window.location.reload();">
                  <source src="data:audio/wav;base64,{b64b}" type="audio/wav">
                </audio>
                ''', unsafe_allow_html=True)

        st.session_state["messages"].append({
            "role": "assistant",
            "content": ans_txt,
            "audio": ans_wav
        })

        st.session_state["recorder_key"] += 1
        st.rerun()

if __name__ == "__main__":
    main()
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py successfully written.")

!pkill -f streamlit || echo "No existing streamlit"
!pkill -f ngrok || echo "No existing ngrok"

print("Starting Streamlit on port 8501...")

proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(5)
print("Connecting ngrok...")

try:
    ngrok.kill()
    tunnel = ngrok.connect(8501, "http")
    print(f"\nNgrok public URL: {tunnel}\n")
    print("Open this link in a new browser tab.")
except Exception as e:
    print("ngrok error:", e)
    proc.terminate()

# Keep cell alive + logs
try:
    while True:
        out_line = proc.stdout.readline()
        if out_line:
            print("[STREAMLIT-LOG]", out_line, end="")
        err_line = proc.stderr.readline()
        if err_line:
            print("[STREAMLIT-ERR]", err_line, end="")
        if proc.poll() is not None:
            print("\nStreamlit ended.")
            break
        time.sleep(1)

except KeyboardInterrupt:
    print("\nKeyboardInterrupt -> shutting down.")
    proc.terminate()
    ngrok.kill()
finally:
    proc.terminate()
    ngrok.kill()
    print("Done.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.
Done installing. Checking Streamlit:
Streamlit version: 1.41.1
app.py successfully written.
^C
^C
Starting Streamlit on port 8501...
Connecting ngrok...

Ngrok public URL: NgrokTunnel: "https://7fab-34-124-235-40.ngrok-free.app" -> "http://localhost:8501"

Open this link in a new browser tab.
[STREAMLIT-LOG] 
[STREAMLIT-ERR] 2025-01-02 13:54:54.758 Uncaught exception GET /_stcore/stream (127.0.0.1)
[STREAMLIT-LOG] Collecting usage statistics. To deactivate, set browser.gatherUsageStats to false.
[STREAMLIT-ERR] HTTPServerRequest(protocol='http', host='7fab-34-124-235-40.ngrok-fr